# BioASQ 14 Task B — Phase B Pipeline (v2, eval-aligned)

End-to-end notebook for **Phase B only** of BioASQ 14 Task B.

**Phase B setup.** You receive a JSON file of questions. Each question has:
- `body` — the natural-language question
- `type` — one of `yesno`, `factoid`, `list`, `summary`
- `documents` — list of gold PubMed URLs (relevant articles)
- `snippets` — gold text spans from those articles, sufficient to answer the question

For each question your system must produce:
- `exact_answer` — `"yes"`/`"no"` for yesno; list-of-lists of up to 5 ranked entities for factoid; list-of-lists of entities for list; **omitted** for summary
- `ideal_answer` — one paragraph, ≤200 words, summarizing the relevant information

**Official Phase B metrics (per BioASQ D4.1):**

| Type | Official | For completeness |
|---|---|---|
| yesno | macro-averaged F1 (maF1) | accuracy |
| factoid | MRR (over top 5) | strict accuracy (SAcc), lenient accuracy (LAcc) |
| list | mean F-measure | mean precision, mean recall |
| ideal | manual scores (1–5 on 4 criteria) | ROUGE-2, ROUGE-SU4 |

Manual scoring is done by BioASQ biomedical experts and can't be reproduced locally. The automatic proxies — ROUGE-2 and ROUGE-SU4 — are implemented in this notebook.

**Note on test data.** As of the current schedule, the test batch JSON may not yet be available. The notebook is structured so the model and the eval pipeline run end-to-end on the development set even if `TEST_PATH` doesn't exist; the test inference block is automatically skipped in that case.

## 1. Install dependencies

Kaggle base images include `torch` and `transformers`. We add quantization, NLTK for tokenization, and `tqdm`. **ROUGE-2 and ROUGE-SU4 are implemented from scratch** in section 11, so no ROUGE package is needed.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes sentencepiece
!pip install -q nltk
import nltk
nltk.download('punkt_tab', quiet=True)

## 2. Imports & setup

In [ ]:
import json
import re
import os
import gc
from pathlib import Path
from collections import Counter, defaultdict
from typing import Any

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

import random
import numpy as np
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

print(f"torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 3. Configuration

Set paths to match your Kaggle dataset layout. **`TEST_PATH` is optional** — if the file doesn't exist, the test-inference cells are skipped automatically and you can still run the local eval on the development set.

In [ ]:
class Config:
    # --- Paths ---
    TRAIN_PATH = "/kaggle/input/datasets/azkaqadir/bioasq-14b/BioASQ-training14b/BioASQ-training14b/training14b.json"

    RUNS = [
        
        {
            "TEST_PATH":       "/kaggle/input/datasets/azkaqadir/bioasq-14b/BioASQ-testing14b/BioASQ-training14b/BioASQ-task14bPhaseB-testset2",
            "SUBMISSION_PATH": "/kaggle/working/submission_batch2.json",
        },
        {
            "TEST_PATH":       "/kaggle/input/datasets/azkaqadir/bioasq-14b/BioASQ-testing14b/BioASQ-training14b/BioASQ-task14bPhaseB-testset3",
            "SUBMISSION_PATH": "/kaggle/working/submission_batch3.json",
        },
        {
            "TEST_PATH":       "/kaggle/input/datasets/azkaqadir/bioasq-14b/BioASQ-testing14b/BioASQ-training14b/BioASQ-task14bPhaseB-testset4",
            "SUBMISSION_PATH": "/kaggle/working/submission_batch4.json",
        },
    ]

    # --- Model ---
    MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
    USE_4BIT = True
    MAX_CONTEXT_SNIPPETS = 12
    SNIPPET_MAX_CHARS = 600
    MAX_NEW_TOKENS_YESNO   = 8
    MAX_NEW_TOKENS_FACTOID = 120
    MAX_NEW_TOKENS_LIST    = 256
    MAX_NEW_TOKENS_IDEAL   = 320
    HOLDOUT_N = 200

    # --- Few-shot ablation ---
    # False = primary system (reported as main results in the paper).
    # True  = TF-IDF few-shot ablation (the variant used for the submitted
    #         test batches). Improves list F1, hurts yesno/factoid/ideal.
    USE_FEW_SHOT = False

cfg = Config()

# Set ACTIVE_RUN = 0, 1, 2, or 3 to switch between runs
ACTIVE_RUN = 0
cfg.TEST_PATH       = cfg.RUNS[ACTIVE_RUN]["TEST_PATH"]
cfg.SUBMISSION_PATH = cfg.RUNS[ACTIVE_RUN]["SUBMISSION_PATH"]

HAS_TEST = os.path.exists(cfg.TEST_PATH)
print(f"TRAIN_PATH exists: {os.path.exists(cfg.TRAIN_PATH)}")
print(f"TEST_PATH exists:  {HAS_TEST}  (test inference will be {'RUN' if HAS_TEST else 'SKIPPED'})")

## 4. Load the training data and inspect it

The development set is shipped as a single JSON file with a top-level `"questions"` array.

In [ ]:
def load_bioasq(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

train = load_bioasq(cfg.TRAIN_PATH)
print(f"Training questions: {len(train['questions'])}")

type_counts = Counter(q["type"] for q in train["questions"])
print("\nQuestion type distribution:")
for t, c in type_counts.most_common():
    print(f"  {t:8s} {c:5d} ({c/len(train['questions'])*100:.1f}%)")

Look at one example of each type so the input/output structure is concrete.

In [ ]:
def preview(q, max_snip=2):
    print(f"id:     {q['id']}")
    print(f"type:   {q['type']}")
    print(f"body:   {q['body']}")
    print(f"#docs:  {len(q.get('documents', []))}")
    print(f"#snips: {len(q.get('snippets', []))}")
    print(f"exact_answer:  {q.get('exact_answer')}")
    ideal = q.get('ideal_answer')
    if isinstance(ideal, list):
        ideal = ideal[0] if ideal else ""
    print(f"ideal_answer:  {(ideal or '')[:220]}{'...' if ideal and len(ideal) > 220 else ''}")
    for s in q.get("snippets", [])[:max_snip]:
        print(f"  snippet: {s['text'][:160]}...")
    print()

for qtype in ["yesno", "factoid", "list", "summary"]:
    example = next(q for q in train["questions"] if q["type"] == qtype)
    print(f"===== {qtype.upper()} =====")
    preview(example)

## 5. Load the LLM

4-bit nf4 quantization so a 3–8B model fits a T4. On P100/A100, set `USE_4BIT = False`.

In [ ]:
quant_config = None
if cfg.USE_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

tokenizer = AutoTokenizer.from_pretrained(cfg.MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    cfg.MODEL_NAME,
    quantization_config=quant_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
)
model.eval()
print(f"Loaded {cfg.MODEL_NAME}")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## 6. Generation helper

Uses the tokenizer's chat template, so it works for any chat-style model (Phi-3, Mistral-Instruct, Qwen, Gemma-it, BioMistral, etc.).

In [ ]:
@torch.no_grad()
def llm_generate(prompt: str, max_new_tokens: int = 128, temperature: float = 0.0) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)

    output = model.generate(
        input_ids,
        max_new_tokens=max_new_tokens,
        do_sample=temperature > 0,
        temperature=max(temperature, 1e-5),
        top_p=1.0 if temperature == 0 else 0.9,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    response = tokenizer.decode(
        output[0][input_ids.shape[-1]:],
        skip_special_tokens=True,
    )
    return response.strip()

print(llm_generate("Say only the word OK.", max_new_tokens=4))

## 7. Build per-question context from gold snippets

In [ ]:
def build_context(question: dict) -> str:
    snippets = question.get("snippets", []) or []
    snippets = snippets[: cfg.MAX_CONTEXT_SNIPPETS]
    parts = []
    for i, s in enumerate(snippets, 1):
        text = (s.get("text") or "").strip().replace("\n", " ")
        if len(text) > cfg.SNIPPET_MAX_CHARS:
            text = text[: cfg.SNIPPET_MAX_CHARS] + "..."
        parts.append(f"[{i}] {text}")
    return "\n".join(parts) if parts else "(no snippets provided)" 

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build per-type TF-IDF index on training questions (excluding holdout)
train_questions = train["questions"][:-cfg.HOLDOUT_N]
type_index = {}
for qtype in ["yesno", "factoid", "list", "summary"]:
    typed = [q for q in train_questions if q["type"] == qtype]
    if not typed:
        continue
    bodies = [q["body"] for q in typed]
    vec = TfidfVectorizer(stop_words="english", max_features=5000)
    matrix = vec.fit_transform(bodies)
    type_index[qtype] = {"questions": typed, "vectorizer": vec, "matrix": matrix}
print("TF-IDF index built:", {k: len(v["questions"]) for k, v in type_index.items()})

def get_few_shot(question: dict, n: int = 3) -> list:
    # Primary system: no few-shot examples. All four answer heads already
    # treat an empty list as "no examples block", so this single guard
    # switches the entire pipeline between primary and ablation.
    if not getattr(cfg, "USE_FEW_SHOT", False):
        return []
    qtype = question["type"]
    if qtype not in type_index:
        return []
    idx = type_index[qtype]
    q_vec = idx["vectorizer"].transform([question["body"]])
    sims = cosine_similarity(q_vec, idx["matrix"]).flatten()
    top = np.argsort(sims)[::-1][:n]
    return [idx["questions"][i] for i in top]

# ── Few-shot example formatters (one per type) ──

def _format_examples_yesno(examples):
    out = []
    for ex in examples:
        ans = ex.get("exact_answer", "yes")
        if isinstance(ans, list): ans = ans[0]
        out.append(f"Q: {ex['body']}\nA: {ans}")
    return "\n\n".join(out)

def _format_examples_factoid(examples):
    out = []
    for ex in examples:
        ea = ex.get("exact_answer", [])
        if ea and isinstance(ea[0], list):
            ans = ea[0][0] if ea[0] else "unknown"
        elif ea:
            ans = ea[0]
        else:
            ans = "unknown"
        out.append(f"Q: {ex['body']}\nA: {ans}")
    return "\n\n".join(out)

def _format_examples_list(examples):
    out = []
    for ex in examples:
        ea = ex.get("exact_answer", [])
        ents = []
        for e in ea[:5]:
            ents.append(e[0] if isinstance(e, list) else e)
        out.append(f"Q: {ex['body']}\nA: {', '.join(ents)}")
    return "\n\n".join(out)

def _format_examples_ideal(examples):
    out = []
    for ex in examples:
        ideal = ex.get("ideal_answer", "")
        if isinstance(ideal, list): ideal = ideal[0] if ideal else ""
        out.append(f"Q: {ex['body']}\nA: {ideal[:300]}")
    return "\n\n".join(out)


## 8. Answer heads — one per question type

### 8.1 Yes/No
Per the eval spec, official metric is **macro-averaged F1** = (F1_yes + F1_no) / 2.

In [ ]:
YESNO_PROMPT = """You are a biomedical expert. Using ONLY the evidence snippets below, answer the question with a single word: "yes" or "no".

Evidence:
{context}

Question: {question}

Answer (one word, yes or no):"""

def answer_yesno(question: dict) -> str:
    examples = _format_examples_yesno(get_few_shot(question, n=2))
    few_shot_block = f"Here are similar solved examples:\n{examples}\n\n" if examples else ""
    prompt = few_shot_block + YESNO_PROMPT.format(
        context=build_context(question),
        question=question["body"],
    )
    out = llm_generate(prompt, max_new_tokens=cfg.MAX_NEW_TOKENS_YESNO).lower()
    m = re.search(r"\b(yes|no)\b", out)
    if m:
        return m.group(1)
    return "yes"


### 8.2 Factoid
Official metric is **MRR over the top-5 ranked candidates**. Ordering is essential.

In [ ]:
FACTOID_PROMPT = """You are a biomedical expert. Using ONLY the evidence snippets below, identify the single entity (or short phrase) that answers the question.

Provide up to 5 distinct candidate answers, ordered from most likely to least likely. Output ONLY the candidates, one per line, with no numbering, no explanation, no preamble. Each candidate should be a short noun phrase (typically 1-5 words), not a sentence.

Evidence:
{context}

Question: {question}

Candidates:"""

def _clean_entity(s: str) -> str:
    s = s.strip()
    s = re.sub(r"^[\-\*\u2022\d]+[\.\)]?\s*", "", s)
    s = s.rstrip(" .,;:")
    s = re.sub(r"\s+", " ", s)
    return s

def answer_factoid(question: dict) -> list:
    examples = _format_examples_factoid(get_few_shot(question, n=3))
    few_shot_block = f"Here are similar solved examples:\n{examples}\n\n" if examples else ""
    prompt = few_shot_block + FACTOID_PROMPT.format(
        context=build_context(question),
        question=question["body"],
    )
    out = llm_generate(prompt, max_new_tokens=cfg.MAX_NEW_TOKENS_FACTOID)
    candidates = []
    seen = set()
    for line in out.split("\n"):
        ent = _clean_entity(line)
        if not ent or len(ent) > 100:
            continue
        key = ent.lower()
        if key in seen:
            continue
        seen.add(key)
        candidates.append(ent)
        if len(candidates) == 5:
            break
    if not candidates:
        candidates = ["unknown"]
    return [[c] for c in candidates]


### 8.3 List
Official metric is **mean F-measure** (set-based F1, with synonym matching done by the BioASQ evaluator).

In [ ]:
LIST_PROMPT = """You are a biomedical expert. Using ONLY the evidence snippets below, identify all entities that answer the question.

Output ONLY the entity names, one per line. No numbering, no explanation, no duplicates. Each entity should be a short noun phrase (typically 1-6 words). Include every distinct entity supported by the snippets, but do not invent or guess entities that are not supported.

Evidence:
{context}

Question: {question}

Entities:"""

def answer_list(question: dict) -> list:
    examples = _format_examples_list(get_few_shot(question, n=3))
    few_shot_block = f"Here are similar solved examples:\n{examples}\n\n" if examples else ""
    prompt = few_shot_block + LIST_PROMPT.format(
        context=build_context(question),
        question=question["body"],
    )
    out = llm_generate(prompt, max_new_tokens=cfg.MAX_NEW_TOKENS_LIST)
    entities = []
    seen = set()
    for line in out.split("\n"):
        ent = _clean_entity(line)
        if not ent or len(ent) > 100:
            continue
        key = ent.lower()
        if key in seen:
            continue
        seen.add(key)
        entities.append(ent)
        if len(entities) == 100:
            break
    if not entities:
        entities = ["unknown"]
    return [[e] for e in entities]


### 8.4 Ideal answer
A single paragraph, ≤200 words. Used for all four question types.

In [ ]:
IDEAL_PROMPT = """You are a biomedical expert. Write a single paragraph that answers the question, using ONLY the evidence snippets below. The paragraph must:
- Be at most 200 words.
- Be a single paragraph (no bullet points, no lists, no headings).
- Synthesize information across snippets in your own words.
- Not include citations like "[1]" or "Snippet 1".
- Not hedge with phrases like "based on the snippets" -- just state the answer.

Evidence:
{context}

Question: {question}

Answer paragraph:"""

def _truncate_words(text: str, max_words: int = 200) -> str:
    import nltk
    sentences = nltk.sent_tokenize(text)
    out, count = [], 0
    for s in sentences:
        w = len(s.split())
        if count + w > max_words:
            break
        out.append(s)
        count += w
    return " ".join(out) if out else " ".join(text.split()[:max_words])

def answer_ideal(question: dict) -> str:
    examples = _format_examples_ideal(get_few_shot(question, n=2))
    few_shot_block = f"Here are similar solved examples:\n{examples}\n\n" if examples else ""
    prompt = few_shot_block + IDEAL_PROMPT.format(
        context=build_context(question),
        question=question["body"],
    )
    out = llm_generate(prompt, max_new_tokens=cfg.MAX_NEW_TOKENS_IDEAL)
    out = re.sub(r"\s+", " ", out).strip()
    out = re.sub(r"^(answer\s*paragraph\s*:\s*)", "", out, flags=re.IGNORECASE)
    out = _truncate_words(out, 200)
    if not out:
        out = "No answer could be produced from the provided snippets."
    return out


## 9. Per-question orchestration

In [ ]:
def process_question(q: dict) -> dict:
    qtype = q["type"]
    record = {
        "id":   q["id"],
        "type": qtype,
        "body": q["body"],
    }
    record["ideal_answer"] = answer_ideal(q)

    if qtype == "yesno":
        record["exact_answer"] = answer_yesno(q)
    elif qtype == "factoid":
        record["exact_answer"] = answer_factoid(q)
    elif qtype == "list":
        record["exact_answer"] = answer_list(q)
    elif qtype == "summary":
        pass   # no exact_answer for summary
    else:
        raise ValueError(f"Unknown question type: {qtype}")

    return record

## 10. Sanity check on a few training questions

In [ ]:
SANITY_N = 2
samples = []
for qtype in ["yesno", "factoid", "list", "summary"]:
    typed = [q for q in train["questions"] if q["type"] == qtype][:SANITY_N]
    samples.extend(typed)

for q in samples:
    pred = process_question(q)
    print(f"===== {q['type'].upper()} | id={q['id']} =====")
    print(f"Q: {q['body']}")
    print(f"GOLD exact:  {q.get('exact_answer')}")
    print(f"PRED exact:  {pred.get('exact_answer')}")
    gold_ideal = q.get('ideal_answer')
    if isinstance(gold_ideal, list):
        gold_ideal = gold_ideal[0] if gold_ideal else ""
    print(f"GOLD ideal:  {(gold_ideal or '')[:200]}...")
    print(f"PRED ideal:  {pred['ideal_answer'][:200]}...")
    print()

## 11. ROUGE-2 and ROUGE-SU4 (implemented from scratch)

The official BioASQ automatic metric for `ideal_answer` is **both ROUGE-2 and ROUGE-SU4**. The widely used `rouge-score` package doesn't implement SU4, so we implement both metrics ourselves.

**Definitions** (per Lin 2004, the reference cited in the BioASQ eval spec):
- **ROUGE-2** — overlap of word bigrams between the system summary and reference(s).
- **ROUGE-SU4** — overlap of *skip-bigrams* (any in-order word pair with a gap of at most 4 words between them) **plus** unigrams.

We use F1 of overlap counts, and when multiple gold references are available (training data's `ideal_answer` is often a list), we take the **max F1 across references** — the standard multi-reference convention.

In [ ]:
_TOK_RE = re.compile(r"[A-Za-z0-9]+")

def _tokenize(text: str) -> list:
    return _TOK_RE.findall((text or "").lower())

def _ngrams(words: list, n: int) -> Counter:
    if len(words) < n:
        return Counter()
    return Counter(tuple(words[i:i+n]) for i in range(len(words) - n + 1))

def _su4_grams(words: list, max_skip_distance: int = 4) -> Counter:
    """Return a Counter of SU4 grams: unigrams + skip-bigrams with at most `max_skip_distance` words between.

    Skip distance d means there are d words between positions i and j, i.e. j - i = d + 1.
    Max skip distance of 4 → j - i ranges from 1 to 5.
    Unigrams and skip-bigrams are kept distinct via tag-prefixed tuples.
    """
    grams = Counter()
    # Unigrams (the 'U' in SU4)
    for w in words:
        grams[("_U", w)] += 1
    # Skip-bigrams
    n = len(words)
    max_offset = max_skip_distance + 1   # j - i can be 1..(max_skip+1)
    for i in range(n):
        for j in range(i + 1, min(i + 1 + max_offset, n)):
            grams[("_S", words[i], words[j])] += 1
    return grams

def _prf_from_counters(pred_c: Counter, ref_c: Counter):
    if not pred_c or not ref_c:
        return 0.0, 0.0, 0.0
    overlap = sum((pred_c & ref_c).values())
    p_total = sum(pred_c.values())
    r_total = sum(ref_c.values())
    precision = overlap / p_total if p_total else 0.0
    recall    = overlap / r_total if r_total else 0.0
    if precision + recall == 0:
        return 0.0, 0.0, 0.0
    f1 = 2 * precision * recall / (precision + recall)
    return precision, recall, f1

def _normalize_refs(refs):
    """refs may be a string or a list of strings; return a list of non-empty strings."""
    if refs is None:
        return []
    if isinstance(refs, str):
        return [refs] if refs.strip() else []
    return [r for r in refs if isinstance(r, str) and r.strip()]

def rouge_2(pred: str, refs) -> float:
    """ROUGE-2 F1 against multiple references — take max."""
    refs = _normalize_refs(refs)
    if not refs or not pred:
        return 0.0
    pred_c = _ngrams(_tokenize(pred), 2)
    best = 0.0
    for r in refs:
        ref_c = _ngrams(_tokenize(r), 2)
        _, _, f1 = _prf_from_counters(pred_c, ref_c)
        if f1 > best:
            best = f1
    return best

def rouge_su4(pred: str, refs) -> float:
    """ROUGE-SU4 F1 against multiple references — take max."""
    refs = _normalize_refs(refs)
    if not refs or not pred:
        return 0.0
    pred_c = _su4_grams(_tokenize(pred))
    best = 0.0
    for r in refs:
        ref_c = _su4_grams(_tokenize(r))
        _, _, f1 = _prf_from_counters(pred_c, ref_c)
        if f1 > best:
            best = f1
    return best

# Quick self-test
ref_test = "The cat sat on the mat"
pred_test = "A cat sat on the mat today"
print(f"Self-test ROUGE-2   : {rouge_2(pred_test, [ref_test]):.4f}")
print(f"Self-test ROUGE-SU4 : {rouge_su4(pred_test, [ref_test]):.4f}")

## 12. Evaluation functions aligned to the BioASQ D4.1 spec

For each question type we compute the **official** metric plus the "for completeness" metrics named in the spec.

In [ ]:
def _norm(s: str) -> str:
    return re.sub(r"[^a-z0-9]+", " ", (s or "").lower()).strip()

# -----------------------------
# Yes/No: macro-F1 (OFFICIAL) + accuracy (completeness)
# -----------------------------
def eval_yesno(preds, golds):
    n = len(preds)
    if n == 0:
        return {}
    # accuracy
    correct = sum(1 for p, g in zip(preds, golds) if p == g)
    acc = correct / n

    # per-class F1
    tp = {"yes": 0, "no": 0}
    fp = {"yes": 0, "no": 0}
    fn = {"yes": 0, "no": 0}
    for p, g in zip(preds, golds):
        for cls in ("yes", "no"):
            if p == cls and g == cls: tp[cls] += 1
            elif p == cls and g != cls: fp[cls] += 1
            elif p != cls and g == cls: fn[cls] += 1
    f1 = {}
    for cls in ("yes", "no"):
        prec = tp[cls] / (tp[cls] + fp[cls]) if (tp[cls] + fp[cls]) else 0.0
        rec  = tp[cls] / (tp[cls] + fn[cls]) if (tp[cls] + fn[cls]) else 0.0
        f1[cls] = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
    macro_f1 = (f1["yes"] + f1["no"]) / 2
    return {
        "n": n,
        "accuracy": acc,             # completeness
        "F1_yes":   f1["yes"],
        "F1_no":    f1["no"],
        "macro_F1": macro_f1,        # OFFICIAL
    }

# -----------------------------
# Factoid: MRR (OFFICIAL) + SAcc, LAcc (completeness)
# -----------------------------
def eval_factoid(preds, gold_synonym_sets):
    """preds: list of [[ent1],[ent2],...] (up to 5);  gold_synonym_sets: list of sets of normalized synonym strings."""
    n = len(preds)
    if n == 0:
        return {}
    rr_total = 0.0
    strict_correct = 0
    lenient_correct = 0
    for pred_list, gold_set in zip(preds, gold_synonym_sets):
        rr = 0.0
        for rank, inner in enumerate(pred_list, 1):
            if rank > 5:
                break
            if inner and _norm(inner[0]) in gold_set:
                rr = 1.0 / rank
                if rank == 1:
                    strict_correct += 1
                lenient_correct += 1
                break
        rr_total += rr
    return {
        "n": n,
        "strict_accuracy":  strict_correct / n,   # completeness
        "lenient_accuracy": lenient_correct / n,  # completeness
        "MRR":              rr_total / n,         # OFFICIAL
    }

# -----------------------------
# List: mean F1 (OFFICIAL) + mean P, mean R (completeness)
# -----------------------------
def eval_list(preds, gold_sets):
    n = len(preds)
    if n == 0:
        return {}
    p_total = r_total = f_total = 0.0
    for pred_list, gold_set in zip(preds, gold_sets):
        pred_set = {_norm(x[0]) for x in pred_list if x}
        if not pred_set and not gold_set:
            p_total += 1.0; r_total += 1.0; f_total += 1.0; continue
        tp = len(pred_set & gold_set)
        prec = tp / len(pred_set) if pred_set else 0.0
        rec  = tp / len(gold_set) if gold_set else 0.0
        f1   = 2*prec*rec/(prec+rec) if (prec+rec) else 0.0
        p_total += prec; r_total += rec; f_total += f1
    return {
        "n": n,
        "mean_precision": p_total / n,  # completeness
        "mean_recall":    r_total / n,  # completeness
        "mean_F1":        f_total / n,  # OFFICIAL
    }

# -----------------------------
# Ideal: ROUGE-2, ROUGE-SU4 (automatic proxies; manual is the true official)
# -----------------------------
def eval_ideal(preds, gold_refs_list):
    """preds: list of strings; gold_refs_list: list of (string or list-of-strings) gold ideal answers."""
    n = len(preds)
    if n == 0:
        return {}
    r2_total = 0.0
    su4_total = 0.0
    for p, g in zip(preds, gold_refs_list):
        r2_total  += rouge_2(p, g)
        su4_total += rouge_su4(p, g)
    return {
        "n": n,
        "ROUGE_2":   r2_total / n,   # OFFICIAL (automatic proxy)
        "ROUGE_SU4": su4_total / n,  # OFFICIAL (automatic proxy)
    }

## 13. Run the local eval on the last `HOLDOUT_N` training questions

In [ ]:
from tqdm.auto import tqdm

dev_questions = train["questions"][-cfg.HOLDOUT_N:]

# Collect predictions and golds, bucketed by type
yesno_preds, yesno_golds = [], []
factoid_preds, factoid_golds = [], []
list_preds, list_golds = [], []
ideal_preds, ideal_golds = [], []   # ideal: all four types

for q in tqdm(dev_questions, desc="Local eval"):
    pred = process_question(q)
    qtype = q["type"]

    # Ideal answer: collect for all four types; use the full list of gold ideal answers (training data has multiple)
    ideal_preds.append(pred["ideal_answer"])
    gold_ideal = q.get("ideal_answer")
    if isinstance(gold_ideal, list):
        ideal_golds.append(gold_ideal)
    elif isinstance(gold_ideal, str):
        ideal_golds.append([gold_ideal])
    else:
        ideal_golds.append([])

    if qtype == "yesno":
        yesno_preds.append(pred["exact_answer"])
        yesno_golds.append((q.get("exact_answer") or "").lower())

    elif qtype == "factoid":
        factoid_preds.append(pred["exact_answer"])
        # Gold factoid in training is a flat list of synonyms for one entity
        gold = q.get("exact_answer") or []
        if gold and isinstance(gold[0], list):
            gold_synonyms = {_norm(s) for inner in gold for s in inner}
        else:
            gold_synonyms = {_norm(s) for s in gold}
        factoid_golds.append(gold_synonyms)

    elif qtype == "list":
        list_preds.append(pred["exact_answer"])
        gold = q.get("exact_answer") or []
        gold_set = set()
        for inner in gold:
            if isinstance(inner, list):
                gold_set.update(_norm(s) for s in inner)
            else:
                gold_set.add(_norm(inner))
        list_golds.append(gold_set)

# Compute metrics
results = {
    "yesno":   eval_yesno(yesno_preds, yesno_golds)     if yesno_preds   else {},
    "factoid": eval_factoid(factoid_preds, factoid_golds) if factoid_preds else {},
    "list":    eval_list(list_preds, list_golds)        if list_preds    else {},
    "ideal":   eval_ideal(ideal_preds, ideal_golds)     if ideal_preds   else {},
}

In [ ]:
import pickle
with open("/kaggle/working/dev_predictions.pkl", "wb") as f:
    pickle.dump({
        "yesno_preds": yesno_preds, "yesno_golds": yesno_golds,
        "factoid_preds": factoid_preds, "factoid_golds": factoid_golds,
        "list_preds": list_preds, "list_golds": list_golds,
        "ideal_preds": ideal_preds, "ideal_golds": ideal_golds,
    }, f)
print("Predictions saved.")

Pretty-print the eval report, clearly marking **OFFICIAL** vs *completeness* metrics.

In [ ]:
import pandas as pd
rows = []
for q, pred_ideal in zip(dev_questions, ideal_preds):
    gold = q.get("ideal_answer")
    if isinstance(gold, list): gold = gold[0]
    rows.append({
        "id": q["id"], "type": q["type"], "question": q["body"],
        "pred_ideal": pred_ideal, "gold_ideal": gold,
    })
pd.DataFrame(rows).to_csv("/kaggle/working/dev_qualitative.csv", index=False)
print("Saved dev_qualitative.csv")

In [ ]:
OFFICIAL_KEYS = {
    "yesno":   {"macro_F1"},
    "factoid": {"MRR"},
    "list":    {"mean_F1"},
    "ideal":   {"ROUGE_2", "ROUGE_SU4"},  # automatic proxies; manual is the true official
}

print("=" * 70)
print(" BioASQ Phase B — Local Evaluation Report")
print("=" * 70)
for qtype, metrics in results.items():
    if not metrics:
        print(f"\n[{qtype}]  no samples in holdout")
        continue
    n = metrics.pop("n", "?")
    print(f"\n[{qtype}]  n = {n}")
    for k, v in metrics.items():
        marker = "  ★ OFFICIAL" if k in OFFICIAL_KEYS.get(qtype, set()) else "    completeness"
        print(f"  {k:18s} {v:.4f}   {marker}")

print("\nNote: for `ideal`, the true OFFICIAL metric is MANUAL expert scoring on 4 criteria")
print("      (info recall, info precision, info repetition, readability — each 1–5).")
print("      ROUGE-2 and ROUGE-SU4 are the official AUTOMATIC proxies and are shown here.")

## 14. Run inference on the test batch (only if the file exists)

If the test JSON is not yet published, this section is skipped — you can still rerun the eval cells above to iterate on prompts / models.

In [ ]:
if HAS_TEST:
    test = load_bioasq(cfg.TEST_PATH)
    print(f"Test questions: {len(test['questions'])}")
    print(f"Type distribution: {Counter(q['type'] for q in test['questions'])}")
else:
    test = None
    print(f"TEST_PATH not found ({cfg.TEST_PATH}). Skipping test-batch inference.")
    print("When the BioASQ test batch is released, set cfg.TEST_PATH to its location and rerun this cell.")

In [ ]:
HAS_TEST = os.path.exists(cfg.TEST_PATH)
print(f"HAS_TEST: {HAS_TEST}")

In [ ]:
if HAS_TEST:
    predictions = []
    for q in tqdm(test["questions"], desc="Predicting test batch"):
        try:
            rec = process_question(q)
        except Exception as e:
            print(f"[warn] question {q.get('id')} failed: {e!r} — falling back to defaults")
            rec = {"id": q["id"], "type": q["type"], "body": q["body"],
                   "ideal_answer": "No answer could be produced from the provided snippets."}
            if q["type"] == "yesno":   rec["exact_answer"] = "yes"
            if q["type"] == "factoid": rec["exact_answer"] = [["unknown"]]
            if q["type"] == "list":    rec["exact_answer"] = [["unknown"]]
        predictions.append(rec)
    print(f"Built {len(predictions)} predictions")
else:
    predictions = []
    print("Skipped — no test file.")

## 15. Validate submission against the BioASQ schema

In [ ]:
def validate_submission(records):
    errors = []
    for i, r in enumerate(records):
        if "id" not in r or "type" not in r:
            errors.append(f"[{i}] missing id/type"); continue
        if "ideal_answer" not in r or not isinstance(r["ideal_answer"], str):
            errors.append(f"[{i}/{r['id']}] ideal_answer must be a non-empty string")
        elif len(r["ideal_answer"].split()) > 200:
            errors.append(f"[{i}/{r['id']}] ideal_answer exceeds 200 words ({len(r['ideal_answer'].split())} words)")

        t = r["type"]
        if t == "yesno":
            if r.get("exact_answer") not in ("yes", "no"):
                errors.append(f"[{i}/{r['id']}] yesno exact_answer must be 'yes' or 'no', got {r.get('exact_answer')!r}")
        elif t == "factoid":
            ea = r.get("exact_answer")
            if not (isinstance(ea, list) and 1 <= len(ea) <= 5 and all(isinstance(x, list) and len(x) == 1 and isinstance(x[0], str) for x in ea)):
                errors.append(f"[{i}/{r['id']}] factoid exact_answer must be list of up to 5 single-element lists")
        elif t == "list":
            ea = r.get("exact_answer")
            if not (isinstance(ea, list) and 1 <= len(ea) <= 100 and all(isinstance(x, list) and len(x) == 1 and isinstance(x[0], str) and len(x[0]) <= 100 for x in ea)):
                errors.append(f"[{i}/{r['id']}] list exact_answer must be list of up to 100 single-element lists, each ≤100 chars")
        elif t == "summary":
            if "exact_answer" in r:
                errors.append(f"[{i}/{r['id']}] summary must NOT have exact_answer")
        else:
            errors.append(f"[{i}/{r['id']}] unknown type {t!r}")
    return errors

if predictions:
    errs = validate_submission(predictions)
    if errs:
        print(f"FOUND {len(errs)} VALIDATION ERRORS:")
        for e in errs[:20]:
            print(" ", e)
    else:
        print("OK — submission passes schema validation.")
else:
    print("No predictions to validate (no test file).")

In [ ]:
if predictions:
    os.makedirs(os.path.dirname(cfg.SUBMISSION_PATH), exist_ok=True)
    with open(cfg.SUBMISSION_PATH, "w", encoding="utf-8") as f:
        json.dump({"questions": predictions}, f, ensure_ascii=False, indent=2)
    print(f"Wrote {cfg.SUBMISSION_PATH} ({os.path.getsize(cfg.SUBMISSION_PATH)/1024:.1f} KB)")
else:
    print("No submission written.")

In [ ]:
for run in cfg.RUNS:
    print(f"\n{'='*60}")
    print(f"Processing: {run['TEST_PATH']}")
    print(f"{'='*60}")
    
    if not os.path.exists(run['TEST_PATH']):
        print(f"File not found, skipping.")
        continue

    test = load_bioasq(run['TEST_PATH'])
    print(f"Questions: {len(test['questions'])}, Types: {Counter(q['type'] for q in test['questions'])}")

    predictions = []
    for q in tqdm(test["questions"], desc=f"Predicting {run['SUBMISSION_PATH'].split('/')[-1]}"):
        try:
            rec = process_question(q)
        except Exception as e:
            print(f"[warn] {q.get('id')} failed: {e!r}")
            rec = {"id": q["id"], "type": q["type"], "body": q["body"],
                   "ideal_answer": "No answer could be produced from the provided snippets."}
            if q["type"] == "yesno":   rec["exact_answer"] = "yes"
            if q["type"] == "factoid": rec["exact_answer"] = [["unknown"]]
            if q["type"] == "list":    rec["exact_answer"] = [["unknown"]]
        predictions.append(rec)

    errs = validate_submission(predictions)
    if errs:
        print(f"VALIDATION ERRORS: {errs[:5]}")
    else:
        print("Validation: OK")
        with open(run['SUBMISSION_PATH'], "w", encoding="utf-8") as f:
            json.dump({"questions": predictions}, f, ensure_ascii=False, indent=2)
        print(f"Saved: {run['SUBMISSION_PATH']} ({os.path.getsize(run['SUBMISSION_PATH'])/1024:.1f} KB)")

## 16. Where to improve

1. **Swap the LLM.** Phi-3-mini is fine as a baseline; BioMistral-7B, Mistral-7B-Instruct-v0.3, and Qwen2.5-7B-Instruct usually outperform it on biomedical QA. Compare with the local eval block above.
2. **Self-consistency for factoid/list.** Sample 5× at `temperature=0.7` and aggregate. Often adds several MRR/F1 points.
3. **Entity normalization.** Use scispaCy (`en_core_sci_md` + `EntityLinker`) or UMLS lookup to canonicalize candidates before deduplication — directly helps list F1 and lenient factoid accuracy.
4. **Fine-tuned yes/no classifier.** A PubMedBERT/BioLinkBERT head on the BioASQ yesno training subset is small and often beats prompting on macro-F1. Ensemble with the LLM via majority vote.
5. **Few-shot prompts.** For each test question, retrieve 2–3 in-context examples from training data via TF-IDF + same-type filter. Lifts all four metrics a few points.
6. **Better truncation for ideal answers.** Stop at the last full sentence within the 200-word cap instead of mid-sentence. Helps both ROUGE and readability scores.
7. **Multi-reference exploit.** Where the gold has multiple ideal answers, generate multiple candidates and pick the one with highest ROUGE-2 + ROUGE-SU4 against the training-data answers from the same question (only valid for self-checking on dev, not on the test batch).

Run order: eval cells above are now diagnostic. Iterate on prompts/models, watch the OFFICIAL metric for each type rise on the holdout, then run the test inference when the test file lands.